# RBDS running, 39 runners vs their own Visual3D: our c3d pipeline on someone else's raw data

Chapter 4, second page. Prev: [nine subjects walking vs Visual3D](validation_report.ipynb) · Next: [the Asics reanalysis from raw c3d](asics_running_c3d.ipynb)

The nine walkers came to us already processed by Visual3D. This one is harder (and fairer, I think): the RBDS running dataset (Fukuchi, Fukuchi & Duarte 2017, PeerJ 5:e3298, CC-BY) ships the RAW c3d files AND the authors' own Visual3D output. So I push their raw markers and forces through the exact c3d calls we use for Asics (static-trial calibration, virtual joint centres, per-bout foot attribution, Newton-Euler about the lab origin) and score the right leg against their answer, at 2.5/3.5/4.5 m/s.

This notebook reports on the following key findings about our reanalysis:

1. **Torques agree about as well as two independent implementations can.** 39 subjects, 101 subject-speeds: median torque RMSE 0.23/0.26/0.11 N m/kg (ankle/knee/hip), median correlation 0.98/0.99/0.99, median energy imbalance 1.6 % of peak dE/dt.
2. **The power gap is mostly about what "joint power" means.** Our wrench power (τ·ω plus the F·Δv translational term) gives power RMSE 1.95/2.04/2.31 W/kg; switching to their τ·ω-only definition drops it to 1.47/1.43/0.86 W/kg. The hip gap mostly goes away.
3. **Their 10 Hz filters and Dempster segment masses barely matter.** Torque RMSE moves to 0.237/0.276/0.109 and 0.221/0.249/0.101 N m/kg, against 0.230/0.257/0.106 for ours.
4. **Their own hip landmark makes our hip torque agree *less*, and no convention closes the torque gap.** Bell (1990) hip takes hip torque RMSE from 0.106 to 0.113 N m/kg; all four of their conventions together give 0.233/0.265/0.127. What's left is the between-implementation floor.

The numbers above are all recomputed from the raw recordings every time this runs (cached in `~/.cache/boneid/rbds`, ~300 MB on first download), so expect it to take a while.

In [1]:
from pathlib import Path
from IPython.display import HTML, display
import json
import numpy as np
from boneid import validate as V
ROOT = Path.cwd().parent
# Recompute every displayed comparison from raw cached recordings.
# Reuse identical variants within this execution, never across backend builds.
runs = {}
def run(variant):
    key = json.dumps(variant, sort_keys=True)
    if key not in runs:
        print(f"Computing RBDS variant: {variant or 'baseline'}", flush=True)
        result = V.rbds_sweep(variant=variant, verbose=False)
        errors = {sid: r["error"] for sid, r in result["results"].items() if "error" in r}
        assert not errors, errors
        runs[key] = result
    return runs[key]
results = run({})
ablations = {name: {"variant": variant, "summary": V.rbds_ablation_summary(run(variant))}
             for name, variant in V.RBDS_VARIANTS.items()}
cumulative = {"steps": [], "theirs": None}
for name, variant in V.RBDS_CUMULATIVE:
    res = run(variant)
    rows = [d for r in res["results"].values()
            for speed, d in r.get("speeds", {}).items()
            if speed == "35" and "error" not in d]
    if cumulative["theirs"] is None:
        cumulative["theirs"] = {
            "torque": np.mean([d["torque_theirs"] for d in rows], axis=0).tolist(),
            "power": np.mean([d["power_theirs"] for d in rows], axis=0).tolist()}
    cumulative["steps"].append({
        "name": name, "variant": variant, "summary": V.rbds_ablation_summary(res),
        "torque_mean_35": (np.mean([d["torque_ours"] for d in rows], axis=0)
                           * V.RBDS_TORQUE_SIGN).tolist(),
        "power_mean_35": np.mean([d["power_ours"] for d in rows], axis=0).tolist()})
display(HTML(V.rbds_report_html(results, ablations=ablations, cumulative=cumulative)))


Computing RBDS variant: baseline


/Users/jeremy/Git/brent/bonelab_inverse_dynamics/src/boneid/io_c3d.py:830: RuntimeWarning: Mean of empty slice
  mean = {k: np.nanmean(v, axis=0) for k, v in static.markers.items()}


/Users/jeremy/Git/brent/bonelab_inverse_dynamics/src/boneid/io_c3d.py:830: RuntimeWarning: Mean of empty slice
  mean = {k: np.nanmean(v, axis=0) for k, v in static.markers.items()}


Computing RBDS variant: {'hip': 'bell'}


/Users/jeremy/Git/brent/bonelab_inverse_dynamics/src/boneid/io_c3d.py:830: RuntimeWarning: Mean of empty slice
  mean = {k: np.nanmean(v, axis=0) for k, v in static.markers.items()}


...

/Users/jeremy/Git/brent/bonelab_inverse_dynamics/src/boneid/io_c3d.py:830: RuntimeWarning: Mean of empty slice
  mean = {k: np.nanmean(v, axis=0) for k, v in static.markers.items()}


/Users/jeremy/Git/brent/bonelab_inverse_dynamics/src/boneid/io_c3d.py:830: RuntimeWarning: Mean of empty slice
  mean = {k: np.nanmean(v, axis=0) for k, v in static.markers.items()}


Computing RBDS variant: {'power': 'scalar', 'hip': 'bell', 'segments': 'dempster'}


/Users/jeremy/Git/brent/bonelab_inverse_dynamics/src/boneid/io_c3d.py:830: RuntimeWarning: Mean of empty slice
  mean = {k: np.nanmean(v, axis=0) for k, v in static.markers.items()}


/Users/jeremy/Git/brent/bonelab_inverse_dynamics/src/boneid/io_c3d.py:830: RuntimeWarning: Mean of empty slice
  mean = {k: np.nanmean(v, axis=0) for k, v in static.markers.items()}
